In [33]:
from matplotlib.colors import XKCD_COLORS
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df_sensores = pd.read_json('/content/sensores.json')
df_qualidade = pd.read_parquet('/content/qualidade.parquet')
df_producao = pd.read_csv('/content/producao.csv')
df_manutencao = pd.read_excel('/content/manutencao.xlsx')

# ETAPA 1

def diagnostico(df, nome):
    print(f'== {nome} ==')
    print(f'Quantidade de linhas e colunas: {df.shape}')
    print(f'Primeiras linhas: {df.head()}')
    print(f'Tipos de dados: {df.dtypes}')
    print(f'Nomes colunas: {df.columns}')
    print(f'Valores ausentes: {df.isna().sum()}')
    print(f'Estísticas básicas: {df.describe()}')
    for col in df.select_dtypes(include="object").columns:
        print(col, "->", df[col].unique())



In [41]:
#Etapa 2
import numpy as np
import pandas as pd

# Carregamento dos dados
df_sensores = pd.read_json("/content/sensores.json")
df_qualidade = pd.read_parquet("/content/qualidade.parquet")
df_producao = pd.read_csv("/content/producao.csv")
df_manutencao = pd.read_excel("/content/manutencao.xlsx")

# Desaninhamento da coluna 'medicoes'
medicoes = pd.json_normalize(df_sensores["medicoes"])
df_sensores = pd.concat(
    [df_sensores.drop(columns="medicoes"), medicoes], axis=1
)

# Conversão de Tipos e Datas
df_producao["data"] = pd.to_datetime(
    df_producao["data"],
    dayfirst=True,
    format='mixed'
)

df_sensores["timestamp"] = pd.to_datetime(
    df_sensores["timestamp"],
    dayfirst=True
)

df_manutencao["custo_manutencao"] = pd.to_numeric(
    df_manutencao["custo_manutencao"], errors="coerce"
)
df_manutencao["data"] = pd.to_datetime(
    df_manutencao["data"],
    dayfirst=True,
    format='mixed'
)

# Remoção de Duplicatas (linhas 100% iguais)
df_producao = df_producao.drop_duplicates()
df_manutencao = df_manutencao.drop_duplicates()
df_qualidade = df_qualidade.drop_duplicates()
df_sensores = df_sensores.drop_duplicates()

# Remoção de chaves duplicadas (mesmo id, dados diferentes)
df_producao = df_producao.sort_values("data").drop_duplicates(
    subset="id_producao", keep="last"
)
df_qualidade = df_qualidade.sort_values(
    "quantidade_inspecionada", ascending=False
).drop_duplicates(subset="id_producao", keep="first")

#Tratamento de Valores Nulos
df_producao["tempo_producao"] = df_producao["tempo_producao"].fillna(
    df_producao["tempo_producao"].median()
)
df_producao["quantidade_produzida"] = df_producao[
    "quantidade_produzida"
].fillna(df_producao["quantidade_produzida"].median())
df_producao["operador"] = df_producao["operador"].fillna("Não informado")
df_producao["data"] = df_producao["data"].fillna(df_producao["data"].mode()[0])

df_manutencao["tempo_parada"] = df_manutencao["tempo_parada"].fillna(
    df_manutencao["tempo_parada"].median()
)
df_manutencao["custo_manutencao"] = df_manutencao["custo_manutencao"].fillna(
    df_manutencao["custo_manutencao"].median()
)

df_qualidade["quantidade_aprovada"] = df_qualidade[
    "quantidade_aprovada"
].fillna(df_qualidade["quantidade_aprovada"].median())

#Trata Outliers e Inconsistências de Forma Vetorial
df_sensores["temperatura"] = df_sensores["temperatura"].mask(
    df_sensores["temperatura"] > 100, df_sensores["temperatura"].median()
)

df_qualidade["indice_qualidade"] = df_qualidade["indice_qualidade"].mask(
    df_qualidade["indice_qualidade"] > 100,
    df_qualidade["indice_qualidade"].median(),
)

#Correção de valores negativos para positivo via valor absoluto
df_sensores["pressao"] = df_sensores["pressao"].abs()
df_manutencao["custo_manutencao"] = df_manutencao["custo_manutencao"].abs()
df_manutencao["tempo_parada"] = df_manutencao["tempo_parada"].abs()
df_producao["quantidade_produzida"] = df_producao[
    "quantidade_produzida"
].abs()

#Padronização de Textos

#lista de máquinas da fábrica
NOMES_OFICIAIS = [
    "Torno CNC 01", "Torno CNC 02", "Fresadora 01", "Fresadora 02",
    "Prensa Hidráulica 01", "Prensa Hidráulica 02", "Solda Robotizada 01",
    "Solda Robotizada 02", "Injetora Plástica 01", "Extrusora 01",
]

def _normalizar_texto_maquina(texto):
    texto = str(texto).strip().upper()
    texto = " ".join(texto.split())
    return texto

def padronizar_maquina(nome_bagunçado):
    nome_normalizado = _normalizar_texto_maquina(nome_bagunçado)

    if nome_normalizado and nome_normalizado[-1].isdigit():
        penultimo = nome_normalizado[-2] if len(nome_normalizado) >= 2 else ""
        if not penultimo.isdigit():
            nome_normalizado = nome_normalizado[:-1] + "0" + nome_normalizado[-1]

    for nome_oficial in NOMES_OFICIAIS:
        if nome_normalizado == _normalizar_texto_maquina(nome_oficial):
            return nome_oficial

    return " ".join(str(nome_bagunçado).strip().split())

df_producao["maquina"] = df_producao["maquina"].apply(padronizar_maquina)
df_manutencao["maquina"] = df_manutencao["maquina"].apply(padronizar_maquina)
df_qualidade["maquina"] = df_qualidade["maquina"].apply(padronizar_maquina)

df_sensores["status"] = (
    df_sensores["status"].astype(str).str.strip().str.lower()
)
df_manutencao["tipo_manutencao"] = (
    df_manutencao["tipo_manutencao"].astype(str).str.strip().str.title()
)
df_producao["turno"] = (
    df_producao["turno"].str.capitalize()
)
df_producao

,id_producao,data,maquina,produto,quantidade_produzida,quantidade_planejada,tempo_producao,operador,turno
250,PR0001,2025-03-01,Torno CNC 01,Engrenagem Industrial,335.0,350,436.0,Juliana Costa,Noite
299,PR0002,2025-03-01,Torno CNC 02,Carcaça Plástica,678.0,700,398.0,André Nascimento,Manhã
149,PR0009,2025-03-01,Solda Robotizada 01,Carcaça Plástica,608.0,700,367.0,Marcos Pereira,Tarde
209,PR0011,2025-03-01,Solda Robotizada 02,Carcaça Plástica,712.0,700,354.0,Marcos Pereira,Manhã
329,PR0013,2025-03-01,Injetora Plástica 01,Suporte Metálico,538.0,600,382.0,Marcos Pereira,Manhã
...,...,...,...,...,...,...,...,...,...
314,PR0336,2025-03-30,Prensa Hidráulica 02,Engrenagem Industrial,351.0,350,388.0,Juliana Costa,Tarde
130,PR0334,2025-03-30,Prensa Hidráulica 01,Suporte Metálico,548.0,600,417.0,Roberto Alves,Manhã
59,PR0339,2025-03-30,Extrusora 01,Componente Hidráulico,259.0,300,443.0,Marcos Pereira,Noite
3,PR0332,2025-03-30,Fresadora 01,Eixo Automotivo,445.0,480,406.0,Marcos Pereira,Manhã


In [42]:
#Etapa 3

CODIGO_DA_MAQUINA = {
    "Torno CNC 01": "MAQ-01",
    "Torno CNC 02": "MAQ-02",
    "Fresadora 01": "MAQ-03",
    "Fresadora 02": "MAQ-04",
    "Prensa Hidráulica 01": "MAQ-05",
    "Prensa Hidráulica 02": "MAQ-06",
    "Solda Robotizada 01": "MAQ-07",
    "Solda Robotizada 02": "MAQ-08",
    "Injetora Plástica 01": "MAQ-09",
    "Extrusora 01": "MAQ-10",
}

df_producao["id_maquina"] = df_producao["maquina"].map(CODIGO_DA_MAQUINA)
df_manutencao["id_maquina"] = df_manutencao["maquina"].map(CODIGO_DA_MAQUINA)

print("Registros de produção sem id_maquina mapeado:", df_producao["id_maquina"].isna().sum())
print("Registros de manutenção sem id_maquina mapeado:", df_manutencao["id_maquina"].isna().sum())


#Dicionário de dados
dicionario_dados = {
    # TABELA: PRODUÇÃO
    "id_producao": {
        "significado": "Identificador único do lote de produção",
        "tipo": "str",
        "origem": "producao.csv",
        "tratamento_realizado": "Remoção de duplicatas (linhas e chaves)",
    },
    "data": {
        "significado": "Data do registro da produção",
        "tipo": "datetime64",
        "origem": "producao.csv",
        "tratamento_realizado": (
            "Convertido com pd.to_datetime(dayfirst=True, format='mixed'),"
            " preenchimento de nulos com a moda"
        ),
    },
    "maquina": {
        "significado": "Nome oficial padronizado do equipamento",
        "tipo": "str",
        "origem": "produção/manutenção/qualidade",
        "tratamento_realizado": (
            "Padronizado com padronizar_maquina(), comparando contra a lista"
            " oficial de máquinas (NOMES_OFICIAIS)"
        ),
    },
    "id_maquina": {
        "significado": "Código oficial do equipamento, usado para unir com sensores",
        "tipo": "str",
        "origem": "derivado de 'maquina' via CODIGO_DA_MAQUINA",
        "tratamento_realizado": (
            "Mapeado a partir do nome já padronizado, usando o cadastro"
            " oficial de máquinas (dicionário fixo)"
        ),
    },
    "produto": {
        "significado": "Código/nome do produto fabricado",
        "tipo": "str",
        "origem": "producao.csv",
        "tratamento_realizado": "Padronizado com .str.strip().str.title()",
    },
    "quantidade_produzida": {
        "significado": "Total de peças produzidas no lote",
        "tipo": "float",
        "origem": "producao.csv",
        "tratamento_realizado": (
            "Convertido com pd.to_numeric(), nulos preenchidos pela mediana,"
            " valores negativos corrigidos com .abs()"
        ),
    },
    "quantidade_planejada": {
        "significado": "Meta de produção do lote",
        "tipo": "int",
        "origem": "producao.csv",
        "tratamento_realizado": "Validação de tipo de dado",
    },
    "tempo_producao": {
        "significado": "Tempo total de execução do lote (minutos)",
        "tipo": "float",
        "origem": "producao.csv",
        "tratamento_realizado": "Preenchimento de valores ausentes pela mediana",
    },
    "operador": {
        "significado": "Nome do operador responsável",
        "tipo": "str",
        "origem": "producao.csv",
        "tratamento_realizado": "Preenchimento de nulos com 'Não informado'",
    },
    "turno": {
        "significado": "Turno de trabalho (Manhã, Tarde, Noite)",
        "tipo": "str",
        "origem": "producao.csv",
        "tratamento_realizado": "Padronizado com .str.capitalize()",
    },
    # TABELA: QUALIDADE
    "quantidade_inspecionada": {
        "significado": "Quantidade de peças inspecionadas",
        "tipo": "float",
        "origem": "qualidade.parquet",
        "tratamento_realizado": "Nenhum tratamento adicional além da remoção de duplicatas",
    },
    "quantidade_aprovada": {
        "significado": "Peças aprovadas no controle de qualidade",
        "tipo": "float",
        "origem": "qualidade.parquet",
        "tratamento_realizado": "Preenchimento de nulos com a mediana",
    },
    "quantidade_rejeitada": {
        "significado": "Peças rejeitadas no controle de qualidade",
        "tipo": "float",
        "origem": "qualidade.parquet",
        "tratamento_realizado": "Nenhum tratamento adicional além da remoção de duplicatas",
    },
    "indice_qualidade": {
        "significado": "Percentual de aprovação do lote (%)",
        "tipo": "float",
        "origem": "qualidade.parquet",
        "tratamento_realizado": (
            "Valores acima de 100% substituídos pela mediana com .mask()"
        ),
    },
    "meta_qualidade": {
        "significado": "Meta de qualidade definida pela empresa (%)",
        "tipo": "float",
        "origem": "qualidade.parquet",
        "tratamento_realizado": "Nenhum tratamento adicional",
    },
    # TABELA: MANUTENÇÃO
    "id_manutencao": {
        "significado": "Identificador único do evento de manutenção",
        "tipo": "str",
        "origem": "manutencao.xlsx",
        "tratamento_realizado": "Remoção de duplicatas",
    },
    "tipo_manutencao": {
        "significado": "Categoria da intervenção (Preventiva, Corretiva, Preditiva)",
        "tipo": "str",
        "origem": "manutencao.xlsx",
        "tratamento_realizado": "Padronizado com .str.strip().str.title()",
    },
    "tempo_parada": {
        "significado": "Duração do tempo de máquina parada (minutos)",
        "tipo": "float",
        "origem": "manutencao.xlsx",
        "tratamento_realizado": (
            "Convertido com pd.to_numeric(), nulos preenchidos pela mediana,"
            " valores negativos corrigidos com .abs()"
        ),
    },
    "custo_manutencao": {
        "significado": "Custo total da manutenção (R$)",
        "tipo": "float",
        "origem": "manutencao.xlsx",
        "tratamento_realizado": (
            "Convertido com pd.to_numeric(), nulos preenchidos pela mediana,"
            " valores negativos corrigidos com .abs()"
        ),
    },
    # TABELA: SENSORES
    "timestamp": {
        "significado": "Data e hora exatas da leitura do sensor",
        "tipo": "datetime64",
        "origem": "sensores.json",
        "tratamento_realizado": "Convertido com pd.to_datetime()",
    },
    "status": {
        "significado": "Estado operacional da máquina no momento da leitura",
        "tipo": "str",
        "origem": "sensores.json",
        "tratamento_realizado": "Padronizado com .str.strip().str.lower()",
    },
    "consumo_energia": {
        "significado": "Consumo elétrico em kWh",
        "tipo": "float",
        "origem": "sensores.json",
        "tratamento_realizado": "Extraído do objeto aninhado medicoes via pd.json_normalize()",
    },
    "temperatura": {
        "significado": "Temperatura medida no sensor (°C)",
        "tipo": "float",
        "origem": "sensores.json",
        "tratamento_realizado": (
            "Extraído de medicoes; valores acima de 100°C substituídos pela"
            " mediana com .mask()"
        ),
    },
    "vibracao": {
        "significado": "Nível de vibração medido",
        "tipo": "float",
        "origem": "sensores.json",
        "tratamento_realizado": "Extraído do objeto aninhado medicoes via pd.json_normalize()",
    },
    "pressao": {
        "significado": "Pressão do sistema",
        "tipo": "float",
        "origem": "sensores.json",
        "tratamento_realizado": (
            "Extraído de medicoes; valores negativos corrigidos com .abs()"
        ),
    },
    "velocidade": {
        "significado": "Velocidade de operação (RPM)",
        "tipo": "float",
        "origem": "sensores.json",
        "tratamento_realizado": "Extraído do objeto aninhado medicoes via pd.json_normalize()",
    },
}

dicionario_dados

Registros de produção sem id_maquina mapeado: 0
Registros de manutenção sem id_maquina mapeado: 0


{'id_producao': {'significado': 'Identificador único do lote de produção',
  'tipo': 'str',
  'origem': 'producao.csv',
  'tratamento_realizado': 'Remoção de duplicatas (linhas e chaves)'},
 'data': {'significado': 'Data do registro da produção',
  'tipo': 'datetime64',
  'origem': 'producao.csv',
  'tratamento_realizado': "Convertido com pd.to_datetime(dayfirst=True, format='mixed'), preenchimento de nulos com a moda"},
 'maquina': {'significado': 'Nome oficial padronizado do equipamento',
  'tipo': 'str',
  'origem': 'produção/manutenção/qualidade',
  'tratamento_realizado': 'Padronizado com padronizar_maquina(), comparando contra a lista oficial de máquinas (NOMES_OFICIAIS)'},
 'id_maquina': {'significado': 'Código oficial do equipamento, usado para unir com sensores',
  'tipo': 'str',
  'origem': "derivado de 'maquina' via CODIGO_DA_MAQUINA",
  'tratamento_realizado': 'Mapeado a partir do nome já padronizado, usando o cadastro oficial de máquinas (dicionário fixo)'},
 'produto': {'

In [43]:
#Etapa 4 — Integração dos dados


df_producao["id_maquina"] = df_producao["maquina"].map(CODIGO_DA_MAQUINA)
df_manutencao["id_maquina"] = df_manutencao["maquina"].map(CODIGO_DA_MAQUINA)

df_sensores["data"] = df_sensores["timestamp"].dt.normalize()

agregados_sensores = (
    df_sensores
    .groupby(["id_maquina", "data"])
    .agg(
        temperatura_media=("temperatura", "mean"),
        vibracao_media=("vibracao", "mean"),
        pressao_media=("pressao", "mean"),
        velocidade_media=("velocidade", "mean"),
        consumo_energia_medio=("consumo_energia", "mean"),
    )
    .reset_index()
)
print("Sensores agregados por dia:", agregados_sensores.shape)
agregados_sensores.head()

df_sensores["eh_falha"] = df_sensores["status"].eq("falha")
flag_falha_diaria = (
    df_sensores
    .groupby(["id_maquina", "data"])["eh_falha"]
    .any()
    .reset_index()
    .rename(columns={"eh_falha": "teve_falha"})
)
agregados_sensores = agregados_sensores.merge(
    flag_falha_diaria, on=["id_maquina", "data"], how="left"
)
agregados_sensores.head()

#integra produção junto com qualidade
df_base = df_producao.merge(
    df_qualidade.drop(columns=["maquina", "produto"], errors="ignore"),
    on="id_producao",
    how="left",
)
print("Merge produção + qualidade:", len(df_base), "linhas")
print("Quantidade de linhas de df_producao:", len(df_producao))

#integra sensores agregados
df_base = df_base.merge(agregados_sensores, on=["id_maquina", "data"], how="left")

#integra manutenção
resumo_manutencao = (
    df_manutencao
    .groupby(["maquina", "data"])
    .agg(
        houve_manutencao=("id_manutencao", "count"),
        custo_manutencao_dia=("custo_manutencao", "sum"),
        tempo_parada_dia=("tempo_parada", "sum"),
    )
    .reset_index()
)
df_base = df_base.merge(resumo_manutencao, on=["maquina", "data"], how="left")

valores_padrao = {
    "houve_manutencao": 0,
    "custo_manutencao_dia": 0,
    "tempo_parada_dia": 0,
}
df_base = df_base.fillna(valores_padrao)
df_base["houve_manutencao"] = df_base["houve_manutencao"].astype(int)

print("Base integrada final:", df_base.shape)
df_base.head()

Sensores agregados por dia: (310, 7)
Merge produção + qualidade: 340 linhas
Quantidade de linhas de df_producao: 340
Base integrada final: (340, 24)


,id_producao,data,maquina,produto,quantidade_produzida,quantidade_planejada,tempo_producao,operador,turno,id_maquina,...,meta_qualidade,temperatura_media,vibracao_media,pressao_media,velocidade_media,consumo_energia_medio,teve_falha,houve_manutencao,custo_manutencao_dia,tempo_parada_dia
0,PR0001,2025-03-01,Torno CNC 01,Engrenagem Industrial,335.0,350,436.0,Juliana Costa,Noite,MAQ-01,...,95.0,64.75,2.34,4.890,1167.25,11.925,False,1,1226.05,83.0
1,PR0002,2025-03-01,Torno CNC 02,Carcaça Plástica,678.0,700,398.0,André Nascimento,Manhã,MAQ-02,...,95.0,71.85,1.75,4.630,1215.15,9.435,False,0,0.00,0.0
2,PR0009,2025-03-01,Solda Robotizada 01,Carcaça Plástica,608.0,700,367.0,Marcos Pereira,Tarde,MAQ-07,...,95.0,66.10,2.14,4.860,1139.55,13.195,False,0,0.00,0.0
3,PR0011,2025-03-01,Solda Robotizada 02,Carcaça Plástica,712.0,700,354.0,Marcos Pereira,Manhã,MAQ-08,...,NaN,64.20,1.94,4.945,1206.60,13.880,False,0,0.00,0.0
4,PR0013,2025-03-01,Injetora Plástica 01,Suporte Metálico,538.0,600,382.0,Marcos Pereira,Manhã,MAQ-09,...,NaN,62.35,2.60,4.590,1278.40,11.475,False,0,0.00,0.0


In [44]:
#Etapa 5
qtd_registros_prod = df_producao.shape[0]
qtd_registros_base = df_base.shape[0]

print(f"Linhas em df_prod (entrada):     {qtd_registros_prod}")
print(f"Linhas em df_base (após merges): {qtd_registros_base}")

sensores_ausentes = df_base["temperatura_media"].isna().sum()
qualidade_ausente = df_base["indice_qualidade"].isna().sum()
duplicatas_base = df_base.duplicated().sum()

print("Registros sem sensores correspondentes (temperatura ausente):", sensores_ausentes)
print("Registros sem inspeção de qualidade correspondente:", qualidade_ausente)
print("Duplicatas na base final:", duplicatas_base)
print()

integridade_preservada = qtd_registros_prod == qtd_registros_base

if integridade_preservada:
    print("✔ Validação OK: a base integrada preserva a quantidade de registros de produção.")
else:
    print("⚠ ATENÇÃO: a integração alterou a quantidade de linhas! Investigar antes de continuar.")

Linhas em df_prod (entrada):     340
Linhas em df_base (após merges): 340
Registros sem sensores correspondentes (temperatura ausente): 1
Registros sem inspeção de qualidade correspondente: 110
Duplicatas na base final: 0

✔ Validação OK: a base integrada preserva a quantidade de registros de produção.


In [52]:
#Etapa 6 — Engenharia de dados
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = df_base.copy()

# Eficiência de produção (produzido / planejado)
df["eficiencia_producao"] = df["quantidade_produzida"] / df["quantidade_planejada"]

# Cumprimento da meta (True/False)
df["cumpriu_meta"] = df["quantidade_produzida"] >= df["quantidade_planejada"]

# Taxa de rejeição
df["taxa_rejeicao"] = df["quantidade_rejeitada"] / df["quantidade_inspecionada"]

# Consumo de energia por unidade produzida
df["consumo_energia_por_unidade"] = df["consumo_energia_medio"] / df["quantidade_produzida"]

# Custo de manutenção por unidade produzida (por linha/dia)
df["custo_manutencao_por_unidade"] = df["custo_manutencao_dia"] / df["quantidade_produzida"]

# Custo de manutenção por máquina (agregado, guardado à parte)
custo_manutencao_por_maquina = (
    df_manutencao.groupby("maquina")["custo_manutencao"].sum().reset_index()
    .rename(columns={"custo_manutencao": "custo_manutencao_total"})
)

# Produtividade por turno (agregado — média de eficiência por turno)
produtividade_por_turno = (
    df.groupby("turno")["eficiencia_producao"].mean().reset_index()
    .rename(columns={"eficiencia_producao": "eficiencia_media_turno"})
)

# Tempo médio de parada (por máquina, a partir da tabela de manutenção)
tempo_medio_parada = (
    df_manutencao.groupby("maquina")["tempo_parada"].mean().reset_index()
    .rename(columns={"tempo_parada": "tempo_medio_parada"})
)


# Índice de utilização da máquina:
tempo_total = df["tempo_producao"] + df["tempo_parada_dia"]
tempo_total_sem_zero = tempo_total.replace(0, np.nan)
df["indice_utilizacao"] = df["tempo_producao"] / tempo_total_sem_zero

#Métrica própria 1
temperatura_media_por_maquina = (
    df.groupby("maquina")["temperatura_media"].mean().reset_index()
    .rename(columns={"temperatura_media": "temperatura_media_da_maquina"})
)
df = df.merge(temperatura_media_por_maquina, on="maquina", how="left")
df["risco_termico"] = df["temperatura_media"] - df["temperatura_media_da_maquina"]

#Métrica própria 2
custo_do_dia = df["custo_manutencao_dia"] + df["consumo_energia_medio"] * 5
pecas_aprovadas_sem_zero = df["quantidade_aprovada"].replace(0, None)
df["custo_total_por_peca_boa"] = custo_do_dia / pecas_aprovadas_sem_zero

df[["maquina", "eficiencia_producao", "taxa_rejeicao", "risco_termico", "custo_total_por_peca_boa"]].describe()

,eficiencia_producao,taxa_rejeicao,risco_termico,custo_total_por_peca_boa
count,340.000000,230.000000,3.390000e+02,230.000000
mean,0.918451,0.059365,-1.027038e-15,2.082462
std,0.126975,0.047424,3.862342e+00,3.268550
min,0.010000,0.000000,-1.055143e+01,0.180682
25%,0.883333,0.023819,-2.100302e+00,0.281119
50%,0.928571,0.045296,9.142857e-02,0.435084
75%,0.963646,0.081016,1.899698e+00,2.979645
max,1.496667,0.207612,1.814857e+01,21.345662


In [46]:
#Etapa 07

#Qual máquina produz mais?
q1 = df.groupby("maquina")["quantidade_produzida"].sum().sort_values(ascending=False)
print("1. Máquina que mais produz:\n", q1.head(1), "\n")

#Qual máquina apresenta menor eficiência?
q2 = df.groupby("maquina")["eficiencia_producao"].mean().sort_values()
print("2. Máquina com menor eficiência:\n", q2.head(1), "\n")

#Qual máquina possui maior custo de manutenção?
q3 = custo_manutencao_por_maquina.sort_values("custo_manutencao_total", ascending=False)
print("3. Máquina com maior custo de manutenção:\n", q3.head(1), "\n")

#Qual máquina apresenta maior consumo de energia por unidade?
q4 = df.groupby("maquina")["consumo_energia_por_unidade"].mean().sort_values(ascending=False)
print("4. Máquina com maior consumo por unidade:\n", q4.head(1), "\n")

#Qual produto possui maior taxa de rejeição?
q5 = df.groupby("produto")["taxa_rejeicao"].mean().sort_values(ascending=False)
print("5. Produto com maior taxa de rejeição:\n", q5.head(1), "\n")

#Qual turno apresenta melhor produtividade?
q6 = produtividade_por_turno.sort_values("eficiencia_media_turno", ascending=False)
print("6. Turno mais produtivo:\n", q6.head(1), "\n")

#Existe relação entre temperatura e falhas?
q7 = df.groupby("teve_falha")["temperatura_media"].mean()
print("7. Temperatura média em dias com e sem falha:\n", q7.round(2), "\n")

print(df[["temperatura_media", "teve_falha"]].corr())

#Existe relação entre vibração e manutenção?
q8 = df.groupby("maquina").agg(
    vibracao_media_geral=("vibracao_media", "mean"),
    eventos_manutencao=("houve_manutencao", "sum")
)
print("8. Vibração média x quantidade de manutenções por máquina:\n", q8.round(2), "\n")

#Qual máquina apresenta maior tempo de parada?
q9 = tempo_medio_parada.sort_values("tempo_medio_parada", ascending=False)
print("9. Máquina com maior tempo médio de parada:\n", q9.head(1), "\n")

#Qual máquina apresenta maior diferença entre planejado e realizado?
df["diferenca_percentual"] = 1 - df["eficiencia_producao"]
q10 = df.groupby("maquina")["diferenca_percentual"].mean().sort_values(ascending=False)
print("10. Máquina com maior diferença planejado x realizado (%):\n", q10.head(1), "\n")

#Qual produto apresenta melhor índice de qualidade?
q11 = df.groupby("produto")["indice_qualidade"].mean().sort_values(ascending=False)
print("11. Produto com melhor índice de qualidade:\n", q11.head(1), "\n")

#Onde está concentrado o maior custo operacional (manutenção + energia)?
consumo_total_por_maquina = (
    df.groupby("maquina")["consumo_energia_medio"].sum().reset_index()
    .rename(columns={"consumo_energia_medio": "consumo_total"})
)
custo_operacional = custo_manutencao_por_maquina.merge(consumo_total_por_maquina, on="maquina")
custo_operacional["custo_operacional_total"] = (
    custo_operacional["custo_manutencao_total"] + custo_operacional["consumo_total"] * 5
)
q12 = custo_operacional.sort_values("custo_operacional_total", ascending=False)
print("12. Onde está concentrado o maior custo operacional:\n", q12.head(1), "\n")


1. Máquina que mais produz:
 maquina
Torno CNC 02    18264.0
Name: quantidade_produzida, dtype: float64 

2. Máquina com menor eficiência:
 maquina
Prensa Hidráulica 01    0.88451
Name: eficiencia_producao, dtype: float64 

3. Máquina com maior custo de manutenção:
                maquina  custo_manutencao_total
7  Solda Robotizada 02                11491.97 

4. Máquina com maior consumo por unidade:
 maquina
Fresadora 02    0.09833
Name: consumo_energia_por_unidade, dtype: float64 

5. Produto com maior taxa de rejeição:
 produto
Componente Hidráulico    0.062999
Name: taxa_rejeicao, dtype: float64 

6. Turno mais produtivo:
      turno  eficiencia_media_turno
2   tarde                 1.078571 

7. Temperatura média em dias com e sem falha:
 teve_falha
False    65.39
True     88.25
Name: temperatura_media, dtype: float64 

                   temperatura_media  teve_falha
temperatura_media           1.000000    0.419489
teve_falha                  0.419489    1.000000
8. Vibração méd

In [47]:
#Etapa 08
maquina_suspeita = "Solda Robotizada 02"

df_suspeita = df[df["maquina"] == maquina_suspeita].sort_values("data")

# Evolução diária dos indicadores da máquina suspeita
evolucao = df_suspeita.groupby("data").agg(
    temperatura_media=("temperatura_media", "mean"),
    vibracao_media=("vibracao_media", "mean"),
    consumo_medio=("consumo_energia_medio", "mean"),
    producao_total=("quantidade_produzida", "sum"),
    rejeicao_media=("taxa_rejeicao", "mean"),
).reset_index()

print(evolucao)

# Eventos de manutenção dessa máquina, em ordem cronológica
manutencoes_maquina = df_manutencao[df_manutencao["maquina"] == maquina_suspeita]
manutencoes_maquina = manutencoes_maquina[["data", "tipo_manutencao", "motivo", "custo_manutencao"]]
manutencoes_maquina = manutencoes_maquina.sort_values("data")
print(manutencoes_maquina)

         data  temperatura_media  vibracao_media  consumo_medio  \
0  2025-03-01              64.20           1.940         13.880   
1  2025-03-02              61.50           2.090         10.715   
2  2025-03-03              62.30           2.040         11.405   
3  2025-03-04              64.35           1.875         13.020   
4  2025-03-05              69.05           1.785         12.925   
5  2025-03-08              69.55           2.275         10.840   
6  2025-03-09              68.85           2.050         10.920   
7  2025-03-10              68.90           1.960         12.660   
8  2025-03-11              71.10           2.075         12.225   
9  2025-03-12              63.05           2.120         11.220   
10 2025-03-13              64.25           2.590         12.910   
11 2025-03-14              68.15           2.075         11.315   
12 2025-03-15              59.55           2.065         11.540   
13 2025-03-16              59.75           2.425         10.80

In [49]:
#Etapa 09

# 1. Gráfico de barras — produção total por máquina
plt.figure(figsize=(10, 5))
q1_plot = df.groupby("maquina")["quantidade_produzida"].sum().sort_values(ascending=False)
plt.bar(q1_plot.index, q1_plot.values, color="steelblue")
plt.title("Produção total por máquina — Março/2025")
plt.xlabel("Máquina")
plt.ylabel("Unidades produzidas")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("grafico_1_barras_producao.png")
plt.close()

# 2. Gráfico de linha — evolução de um indicador da máquina suspeita (OBRIGATÓRIO p/ Etapa 8)
plt.figure(figsize=(10, 5))
plt.plot(evolucao["data"], evolucao["vibracao_media"], marker="o", label="Vibração média")
plt.title(f"Evolução da vibração — {maquina_suspeita} (Março/2025)")
plt.xlabel("Data")
plt.ylabel("Vibração (unidade do sensor)")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.savefig("grafico_2_linha_vibracao.png")
plt.close()

# 3. Histograma — distribuição da eficiência de produção
plt.figure(figsize=(10, 5))
plt.hist(df["eficiencia_producao"].dropna(), bins=20, color="darkorange", edgecolor="black")
plt.title("Distribuição da eficiência de produção")
plt.xlabel("Eficiência (produzido / planejado)")
plt.ylabel("Frequência")
plt.tight_layout()
plt.savefig("grafico_3_histograma_eficiencia.png")
plt.close()

# 4. Gráfico de dispersão — temperatura x vibração
plt.figure(figsize=(10, 5))
plt.scatter(df["temperatura_media"], df["vibracao_media"], alpha=0.4, s=10)
plt.title("Relação entre temperatura e vibração")
plt.xlabel("Temperatura média (°C)")
plt.ylabel("Vibração média")
plt.tight_layout()
plt.savefig("grafico_4_dispersao_temp_vibracao.png")
plt.close()

# 5. Gráfico adicional — custo de manutenção por máquina (barras horizontais)
plt.figure(figsize=(10, 5))
custo_ord = custo_manutencao_por_maquina.sort_values("custo_manutencao_total")
plt.barh(custo_ord["maquina"], custo_ord["custo_manutencao_total"], color="indianred")
plt.title("Custo total de manutenção por máquina")
plt.xlabel("Custo (R$)")
plt.ylabel("Máquina")
plt.tight_layout()
plt.savefig("grafico_5_custo_manutencao.png")
plt.close()

#6. Painel simples com pelo menos 4 KPIs
fig, axs = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle("Painel de KPIs — TECFOR Industrial (Março/2025)", fontsize=14)

kpi1 = df["eficiencia_producao"].mean() * 100
kpi2 = df["taxa_rejeicao"].mean() * 100
kpi3 = df["custo_manutencao_dia"].sum()
kpi4 = df["tempo_parada_dia"].mean()

axs[0, 0].text(0.5, 0.5, f"{kpi1:.1f}%", fontsize=28, ha="center")
axs[0, 0].set_title("Eficiência média de produção")
axs[0, 0].axis("off")

axs[0, 1].text(0.5, 0.5, f"{kpi2:.1f}%", fontsize=28, ha="center")
axs[0, 1].set_title("Taxa média de rejeição")
axs[0, 1].axis("off")

axs[1, 0].text(0.5, 0.5, f"R$ {kpi3:,.0f}", fontsize=24, ha="center")
axs[1, 0].set_title("Custo total de manutenção")
axs[1, 0].axis("off")

axs[1, 1].text(0.5, 0.5, f"{kpi4:.1f} min", fontsize=28, ha="center")
axs[1, 1].set_title("Tempo médio de parada")
axs[1, 1].axis("off")

plt.tight_layout()
plt.savefig("grafico_6_painel_kpis.png")
plt.close()

In [53]:
#Etapa 10
df.to_parquet("base_industrial_final.parquet")

df_check = pd.read_parquet("base_industrial_final.parquet")
assert df_check.shape == df.shape, "Shape mudou após salvar/reabrir o Parquet!"
assert (df_check.dtypes == df.dtypes).all(), "Algum tipo de dado mudou!"

df_check.to_csv("base_industrial_final.csv", index=False)

# comparação de tamanho entre os dois formatos
import os
tamanho_parquet = os.path.getsize("base_industrial_final.parquet")
tamanho_csv = os.path.getsize("base_industrial_final.csv")

print(f"Tamanho Parquet: {tamanho_parquet / 1024:.1f} KB")
print(f"Tamanho CSV:     {tamanho_csv / 1024:.1f} KB")
print(f"Parquet é {tamanho_csv / tamanho_parquet:.1f}x menor que o CSV equivalente.")

Tamanho Parquet: 50.0 KB
Tamanho CSV:     99.2 KB
Parquet é 2.0x menor que o CSV equivalente.


In [51]:
comparacao_tipos = pd.DataFrame({
    "tipo_antes": df.dtypes,
    "tipo_depois": df_check.dtypes,
})
comparacao_tipos["mudou"] = comparacao_tipos["tipo_antes"] != comparacao_tipos["tipo_depois"]
print(comparacao_tipos[comparacao_tipos["mudou"]])


                  tipo_antes tipo_depois  mudou
indice_utilizacao     object     float64   True


14)Eu escolheria o Parquet, porque ele é melhor para muitos dados. Ocupa menos espaço e é mais rápido para fazer análises. Também consegue manter os tipos dos dados.

O CSV é fácil de usar, mas fica pesado com milhões de registros. JSON é bom para troca de dados, mas não é tão bom para análise. Excel também não é indicado para uma quantidade muito grande de dados.

15) Como organizar meu trabalho

Eu dividiria o projeto em 2 sessões:

1ª sessão: Etapas 1, 2, 3, 4 e 5 — importar, tratar, padronizar, integrar e validar os dados.
2ª sessão: Etapas 6, 7, 8, 9 e 10 — criar métricas, analisar, investigar, fazer os gráficos e exportar.

Conclusão executiva

Os três principais problemas encontrados foram:

começo — aconteceu por causa de modelagem.
meio — teve impacto em integrar os daods.
fim — foi identificado pelos dados de no grafico

Eu recomendo focar primeiro na modelagem, que ja é uma das primeira parte, porque foi onde os dados mostraram o maior impacto. Também seria importante acompanhar os indicadores e fazer manutenção preventiva para evitar que o problema continue.